In [ ]:
import pandas as pd
import numpy as np
import random
import uuid  # Ajout de l'import uuid pour g?n?rer des adresses email uniques

# --- Param?tres G?n?raux ---
N_RECORDS = 40000
FILENAME = "dataset_revenu_marocains.csv"
RANDOM_SEED = 42 # Pour la reproductibilit?
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# --- D?finition des Cat?gories ---
MILIEU_OPTS = ['Urbain', 'Rural']
SEXE_OPTS = ['Homme', 'Femme']
NIVEAU_EDUCATION_OPTS = ['Sans niveau', 'Fondamental', 'Secondaire', 'Sup?rieur']
ETAT_MATRIMONIAL_OPTS = ['C?libataire', 'Mari?', 'Divorc?', 'Veuf']
CSP_OPTS = ['Cadres sup?rieurs', 'Professions interm?diaires', 'Employ?s', 'Ouvriers', 'Agriculteurs', 'Inactifs']
REGION_GEO_OPTS = ['Nord', 'Centre', 'Sud', 'Est', 'Ouest']
SECTEUR_EMPLOI_OPTS = ['Public', 'Priv?', 'Informel']
OUI_NON_OPTS = ['Oui', 'Non']

# --- Proportions pour la g?n?ration (peuvent ?tre ajust?es) ---
P_URBAIN = 0.60 # Proportion cible d'urbains (ajustable)
P_RURAL = 1 - P_URBAIN

# --- Fonctions de G?n?ration des Caract?ristiques (similaires ? v1, avec ajustements si besoin) ---

def generate_age(n):
    return np.random.randint(18, 64, n)

def generate_milieu(n):
    return np.random.choice(MILIEU_OPTS, n, p=[P_URBAIN, P_RURAL])

def generate_sexe(n):
    return np.random.choice(SEXE_OPTS, n, p=[0.52, 0.48])

def generate_niveau_education(n):
    return np.random.choice(NIVEAU_EDUCATION_OPTS, n, p=[0.15, 0.30, 0.35, 0.20]) # Un peu plus de 'Sup?rieur'

def generate_annees_experience(age, niveau_education):
    experience = []
    for a, edu in zip(age, niveau_education):
        min_age_travail = 18
        if edu == 'Sup?rieur': min_age_travail = 23
        elif edu == 'Secondaire': min_age_travail = 20
        
        max_exp = a - min_age_travail
        if max_exp < 0: max_exp = 0
        
        exp = np.random.randint(0, max_exp + 1) if max_exp > 0 else 0
        exp = min(exp, a - 16) if a > 16 else 0
        experience.append(max(0,exp))
    return np.array(experience)

def generate_etat_matrimonial(n, age):
    etats = []
    for a in age:
        if a < 25: etats.append(np.random.choice(ETAT_MATRIMONIAL_OPTS, p=[0.8, 0.15, 0.03, 0.02]))
        elif a < 45: etats.append(np.random.choice(ETAT_MATRIMONIAL_OPTS, p=[0.2, 0.65, 0.1, 0.05]))
        else: etats.append(np.random.choice(ETAT_MATRIMONIAL_OPTS, p=[0.1, 0.55, 0.15, 0.2]))
    return np.array(etats)

def generate_csp(niveau_education, annees_experience, age):
    csps = []
    for edu, exp, a in zip(niveau_education, annees_experience, age):
        if exp < 1 and a < 22 and edu in ['Sans niveau', 'Fondamental']: csps.append('Inactifs')
        elif edu == 'Sup?rieur':
            if exp > 8: csps.append('Cadres sup?rieurs')
            elif exp > 2: csps.append('Professions interm?diaires')
            else: csps.append(np.random.choice(['Employ?s', 'Professions interm?diaires'], p=[0.7,0.3]))
        elif edu == 'Secondaire':
            if exp > 12: csps.append('Professions interm?diaires')
            elif exp > 4: csps.append('Employ?s')
            else: csps.append(np.random.choice(['Ouvriers','Employ?s'], p=[0.7,0.3]))
        elif edu == 'Fondamental':
            if exp > 8: csps.append('Ouvriers')
            else: csps.append(np.random.choice(['Ouvriers', 'Agriculteurs'], p=[0.6,0.4]))
        else: # Sans niveau
            csps.append(np.random.choice(['Ouvriers', 'Agriculteurs', 'Inactifs'], p=[0.35,0.35,0.3]))
    return np.array(csps)

def generate_possession_biens(n, csp_list, milieu_list):
    prop_immo, veh_motor, terr_agri = [], [], []
    for csp, milieu in zip(csp_list, milieu_list):
        p_immo, p_veh, p_agri = 0.05, 0.05, 0.02
        if csp == 'Cadres sup?rieurs': p_immo, p_veh = 0.7, 0.8
        elif csp == 'Professions interm?diaires': p_immo, p_veh = 0.5, 0.6
        elif csp == 'Employ?s': p_immo, p_veh = 0.25, 0.35
        elif csp == 'Ouvriers': p_immo, p_veh = 0.1, 0.15
        elif csp == 'Agriculteurs': p_immo, p_veh, p_agri = 0.3, 0.25, 0.6
        
        if milieu == 'Rural':
            p_veh *= 0.7
            if csp == 'Agriculteurs': p_agri = 0.8
        
        prop_immo.append(np.random.choice(OUI_NON_OPTS, p=[p_immo, 1-p_immo]))
        veh_motor.append(np.random.choice(OUI_NON_OPTS, p=[p_veh, 1-p_veh]))
        terr_agri.append(np.random.choice(OUI_NON_OPTS, p=[p_agri, 1-p_agri]))
    return prop_immo, veh_motor, terr_agri

def generate_region_geographique(n):
    return np.random.choice(REGION_GEO_OPTS, n, p=[0.22, 0.28, 0.15, 0.15, 0.20]) # Nord, Centre, Sud, Est, Ouest

def generate_secteur_emploi(n, csp_list):
    secteurs = []
    for csp in csp_list:
        if csp == 'Inactifs': secteurs.append(np.nan)
        elif csp == 'Agriculteurs': secteurs.append(np.random.choice(['Priv?', 'Informel'], p=[0.2,0.8]))
        elif csp in ['Cadres sup?rieurs', 'Professions interm?diaires']: secteurs.append(np.random.choice(['Public', 'Priv?'], p=[0.35, 0.65]))
        elif csp == 'Employ?s': secteurs.append(np.random.choice(['Public', 'Priv?', 'Informel'], p=[0.3, 0.5, 0.2]))
        else: secteurs.append(np.random.choice(['Priv?', 'Informel'], p=[0.4, 0.6]))
    return np.array(secteurs)

def generate_revenu_secondaire(n, csp_list):
    probs = []
    for csp in csp_list:
        if csp in ['Cadres sup?rieurs', 'Professions interm?diaires']: probs.append(0.35)
        elif csp == 'Agriculteurs': probs.append(0.25)
        elif csp == 'Employ?s': probs.append(0.15)
        else: probs.append(0.05)
    return np.array([np.random.choice(OUI_NON_OPTS, p=[p, 1-p]) for p in probs])

# --- Nouvelle Fonction de G?n?ration du Revenu Annuel ---
def adjust_group_revenus(revenus_group, target_prop, tol=0.01, max_iter=50):
    """
    Applique une transformation puissance sur les revenus du groupe afin que
    la proportion des valeurs inf?rieures ? la moyenne soit proche de target_prop.
    Cette transformation pr?serve l'ordre (monotone) et devrait conserver les relations.
    """
    b_low, b_high = 0.5, 2.0
    b = 1.0
    for _ in range(max_iter):
        trans = np.power(revenus_group, b)
        new_mean = trans.mean()
        prop = (trans < new_mean).mean()
        if abs(prop - target_prop) < tol:
            return trans
        if prop > target_prop:
            b_high = b
        else:
            b_low = b
        b = (b_low + b_high) / 2.0
    return trans  # Au bout de max_iter

def generate_revenu_annuel(df):
    n = len(df)
    revenus = np.zeros(n)
    
    # Masques pour identifier les milieux
    is_urbain = (df['Milieu'] == 'Urbain').values
    is_rural = (df['Milieu'] == 'Rural').values

    # Param?tres de distribution de base distincts - R?duction des sigmas
    base_revenu_log_mean_urbain = np.log(11000)
    base_revenu_log_sigma_urbain = 0.65 # R?duit par rapport ? 0.85
    base_revenu_log_mean_rural = np.log(7000)
    base_revenu_log_sigma_rural = 0.95  # R?duit par rapport ? 1.15

    # G?n?ration des revenus de base sp?cifiques
    if is_urbain.sum() > 0:
        revenus[is_urbain] = np.random.lognormal(mean=base_revenu_log_mean_urbain,
                                                  sigma=base_revenu_log_sigma_urbain,
                                                  size=is_urbain.sum())
    if is_rural.sum() > 0:
        revenus[is_rural] = np.random.lognormal(mean=base_revenu_log_mean_rural,
                                                 sigma=base_revenu_log_sigma_rural,
                                                 size=is_rural.sum())

    # Application des bonus/facteurs
    for i in range(n):
        record = df.iloc[i]
        rev = revenus[i]
        # Niveau d'?ducation - impact augment?
        if record['Niveau_education'] == 'Sup?rieur':
            rev += 25000 * (1 + 0.15 * record['Annees_experience']) # Augmentation du bonus et de l'effet de l'exp?rience
        elif record['Niveau_education'] == 'Secondaire':
            rev += 10000 * (1 + 0.1 * record['Annees_experience'])  # Augmentation du bonus
        elif record['Niveau_education'] == 'Fondamental':
            rev += 3000 * (1 + 0.05 * record['Annees_experience']) # L?g?re augmentation
        else: # Sans niveau
            rev += 750 * (1 + 0.03 * record['Annees_experience'])  # L?g?re augmentation

        # CSP - facteurs augment?s
        csp_factor = {
            'Cadres sup?rieurs': 2.5,         # Augment?
            'Professions interm?diaires': 1.8, # Augment?
            'Employ?s': 1.4,                  # Augment?
            'Ouvriers': 1.0,                  # Ajust?
            'Agriculteurs': 0.9,              # Ajust?
            'Inactifs': 0.3                   # R?duit pour marquer la diff?rence
        }.get(record['CSP'], 1.0)
        rev *= csp_factor

        # Sexe - impact l?g?rement ajust? pour plus de clart?
        rev *= 1.25 if record['Sexe'] == 'Homme' else 0.90

        # R?gion - impact maintenu ou l?g?rement ajust? si n?cessaire
        region_factor = {
            'Centre': 1.1, 'Ouest': 1.1, 'Nord': 1.05,
            'Sud': 0.95, 'Est': 0.95
        }.get(record['Region_geographique'], 1.0)
        rev *= region_factor

        # Secteur Emploi - impact ajust?
        if pd.notna(record['Secteur_emploi']) and record['CSP'] != 'Inactifs':
            secteur_factor = {'Public': 1.1, 'Priv?': 1.15, 'Informel': 0.85}.get(record['Secteur_emploi'], 1.0) # Ajustement des facteurs
            rev *= secteur_factor
        
        # Revenu Secondaire - impact maintenu
        if record['Revenu_secondaire'] == 'Oui':
            rev += 3000
        revenus[i] = rev

    # Calibration des moyennes par milieu
    df_temp = pd.DataFrame({'rev': revenus, 'Milieu': df['Milieu'].values})
    CIBLE_URBAIN = 26988
    CIBLE_RURAL = 12862
    mean_urbain = df_temp.loc[is_urbain, 'rev'].mean() if is_urbain.sum() > 0 else 1
    mean_rural = df_temp.loc[is_rural, 'rev'].mean() if is_rural.sum() > 0 else 1
    
    if is_urbain.sum() > 0 and mean_urbain > 0 : # ?viter la division par z?ro ou NaN
        revenus[is_urbain] *= CIBLE_URBAIN / mean_urbain
    if is_rural.sum() > 0 and mean_rural > 0 : # ?viter la division par z?ro ou NaN
        revenus[is_rural] *= CIBLE_RURAL / mean_rural


    # Plafonnement
    revenus = np.clip(revenus, 300, 600000)
    inactifs = (df['CSP'] == 'Inactifs').values
    revenus[inactifs] = np.minimum(revenus[inactifs], 8000)

    # Ajustement des proportions sous la moyenne
    # Vous pouvez moduler ou d?sactiver cette partie pour observer l'impact sur le R?
    global_target = 0.718
    urbain_target = 0.659
    rural_target = 0.854

    # S'assurer que les sous-groupes ne sont pas vides avant d'appeler adjust_group_revenus
    if revenus.size > 0 :
        revenus = adjust_group_revenus(revenus, global_target)
    if is_urbain.sum() > 0 and revenus[is_urbain].size > 0:
        revenus[is_urbain] = adjust_group_revenus(revenus[is_urbain], urbain_target)
    if is_rural.sum() > 0 and revenus[is_rural].size > 0:
        revenus[is_rural] = adjust_group_revenus(revenus[is_rural], rural_target)


    # Renormalisation des moyennes
    if is_urbain.sum() > 0 and np.mean(revenus[is_urbain]) > 0: # ?viter la division par z?ro
        revenus[is_urbain] *= CIBLE_URBAIN / np.mean(revenus[is_urbain])
    if is_rural.sum() > 0 and np.mean(revenus[is_rural]) > 0: # ?viter la division par z?ro
        revenus[is_rural] *= CIBLE_RURAL / np.mean(revenus[is_rural])

    # Plafonnement final
    revenus = np.clip(revenus, 300, 600000)
    return revenus

# --- Fonctions pour Imperfections (am?lior?es) ---
def add_valeurs_manquantes(df, colonnes_pour_nan, p_nan=0.001):
    """Ajoute des valeurs manquantes (NaN) de mani?re al?atoire dans les colonnes sp?cifi?es."""
    for col in colonnes_pour_nan:
        if col in df.columns:
            mask = np.random.choice([True, False], size=len(df), p=[p_nan, 1-p_nan])
            df.loc[mask, col] = np.nan
    return df

def add_valeurs_aberrantes_age(df, p_aberrant=0.001):
    """Ajoute des valeurs aberrantes dans la colonne 'Age'."""
    n_aberr = int(p_aberrant * len(df))
    valid_indices = df.index.tolist()
    aberr_indices = random.sample(valid_indices, min(n_aberr, len(valid_indices)))
    for idx in aberr_indices:
        df.loc[idx, 'Age'] = np.random.choice([-5, 150])  # Valeurs aberrantes
    return df

def add_valeurs_aberrantes_experience(df, p_aberrant=0.001):
    """Ajoute des valeurs aberrantes dans la colonne 'Annees_experience', en tenant compte de l'?ge."""
    n_aberr = int(p_aberrant * len(df))
    valid_indices = df.index.tolist()
    aberr_indices = random.sample(valid_indices, min(n_aberr, len(valid_indices)))
    for idx in aberr_indices:
        age = df.loc[idx, 'Age']
        df.loc[idx, 'Annees_experience'] = age + np.random.randint(10, 30)  # Exp?rience > Age
    return df

def add_valeurs_aberrantes_possession(df, p_aberrant=0.001):
    """Ajoute des valeurs aberrantes dans les colonnes de possession de biens, en tenant compte de la CSP."""
    n_aberr = int(p_aberrant * len(df))
    valid_indices = df.index.tolist()
    aberr_indices = random.sample(valid_indices, min(n_aberr, len(valid_indices)))
    for idx in aberr_indices:
        csp = df.loc[idx, 'CSP']
        if csp == 'Inactifs':
            df.loc[idx, 'Propriete_immobiliere'] = np.random.choice(['Oui', 'Non'], p=[0.9, 0.1])
            df.loc[idx, 'Vehicule_motorise'] = np.random.choice(['Oui', 'Non'], p=[0.8, 0.2])
            df.loc[idx, 'Terrain_agricole'] = np.random.choice(['Oui', 'Non'], p=[0.1, 0.9])
        elif csp == 'Agriculteurs':
            df.loc[idx, 'Terrain_agricole'] = np.random.choice(['Oui', 'Non'], p=[0.05, 0.95])
    return df

def add_valeurs_aberrantes_revenu(revenus_col, csp_col, p_aberrant=0.0005):
    """Ajoute des valeurs aberrantes dans la colonne 'Revenu_Annuel', en tenant compte de la CSP."""
    n_aberr = int(p_aberrant * len(revenus_col))
    valid_indices = revenus_col[revenus_col.notna()].index.tolist()
    aberr_indices = random.sample(valid_indices, min(n_aberr, len(valid_indices)))
    
    for idx in aberr_indices:
        csp = csp_col.loc[idx]
        if csp == 'Inactifs':
            revenus_col.loc[idx] = np.random.choice([100, 200, 300])
        elif csp == 'Cadres sup?rieurs':
            revenus_col.loc[idx] = np.random.choice([1000000, 1200000, 1500000])
        elif csp == 'Professions interm?diaires':
            revenus_col.loc[idx] = np.random.choice([600000, 700000])
        elif csp == 'Employ?s':
            revenus_col.loc[idx] = np.random.choice([400000, 500000])
        elif csp == 'Ouvriers':
            revenus_col.loc[idx] = np.random.choice([200000, 300000])
        elif csp == 'Agriculteurs':
            revenus_col.loc[idx] = np.random.choice([50000, 60000])
        else:
            revenus_col.loc[idx] = np.random.choice([300000, 400000, 500000])
    return revenus_col

# --- G?n?ration du Dataset  ---
def generate_dataset():
    data = pd.DataFrame()

    data['Age'] = generate_age(N_RECORDS)
    data['Milieu'] = generate_milieu(N_RECORDS)
    data['Sexe'] = generate_sexe(N_RECORDS)
    data['Niveau_education'] = generate_niveau_education(N_RECORDS)
    data['Annees_experience'] = generate_annees_experience(data['Age'], data['Niveau_education'])
    data['Etat_matrimonial'] = generate_etat_matrimonial(N_RECORDS, data['Age'])
    data['CSP'] = generate_csp(data['Niveau_education'], data['Annees_experience'], data['Age'])
    
    prop_immo, veh_motor, terr_agri = generate_possession_biens(N_RECORDS, data['CSP'], data['Milieu'])
    data['Propriete_immobiliere'] = prop_immo
    data['Vehicule_motorise'] = veh_motor
    data['Terrain_agricole'] = terr_agri

    data['Region_geographique'] = generate_region_geographique(N_RECORDS)
    data['Secteur_emploi'] = generate_secteur_emploi(N_RECORDS, data['CSP'])
    data['Revenu_secondaire'] = generate_revenu_secondaire(N_RECORDS, data['CSP'])
    
    data['Revenu_Annuel'] = generate_revenu_annuel(data.copy()) # Utilise la nouvelle fonction

    # Imperfections
    cols_with_nan = ['Etat_matrimonial', 'Secteur_emploi', 'Revenu_secondaire', 
                     'Propriete_immobiliere', 'Vehicule_motorise', 'Terrain_agricole',
                     'Annees_experience'] # CSP peut avoir des Inactifs, pas besoin de NaN explicite
    data = add_valeurs_manquantes(data, cols_with_nan, p_nan=0.001)
    
    # Aberrations sur le revenu (apr?s g?n?ration et NaN)
    revenu_not_na_mask = data['Revenu_Annuel'].notna()
    data.loc[revenu_not_na_mask, 'Revenu_Annuel'] = add_valeurs_aberrantes_revenu(
        data.loc[revenu_not_na_mask, 'Revenu_Annuel'].copy(),
        data.loc[revenu_not_na_mask, 'CSP'].copy()
    )

    # AJOUT : Aberrations sur d'autres colonnes
    data = add_valeurs_aberrantes_age(data, p_aberrant=0.001)
    data = add_valeurs_aberrantes_experience(data, p_aberrant=0.001)
    data = add_valeurs_aberrantes_possession(data, p_aberrant=0.001)

    # Colonne d?riv?e utile pour EDA, mais pas pour mod?lisation directe si Age est d?j? l?
    bins = [18, 25, 45, 60, 100] # 100 pour couvrir les ?ges aberrants
    labels = ['Jeune', 'Adulte', 'Senior', '?g?']
    data['Categorie_age'] = pd.cut(data['Age'], bins=bins, labels=labels, right=False)
    
    # V?rification finale de coh?rence pour Annees_experience vs Age
    mask_exp_incoherent = data['Annees_experience'] > (data['Age'] - 16)
    data.loc[mask_exp_incoherent, 'Annees_experience'] = (data.loc[mask_exp_incoherent, 'Age'] - 16 - np.random.randint(0,2, size=mask_exp_incoherent.sum())).clip(lower=0)
    
    # S'assurer que les inactifs ont bien un revenu bas ou NaN
    inactifs_mask = data['CSP'] == 'Inactifs'
    if 'Revenu_Annuel' in data.columns:
        data.loc[inactifs_mask & data['Revenu_Annuel'].notna(), 'Revenu_Annuel'] = \
            data.loc[inactifs_mask & data['Revenu_Annuel'].notna(), 'Revenu_Annuel'].apply(lambda x: min(x, 12000))

    # AJOUT: Colonnes Redondantes et Non Pertinentes
    data['Revenu_Mensuel'] = (data['Revenu_Annuel'] / 12).round(2)
    data['Adresse_Email'] = [f"user{uuid.uuid4().hex[:6]}@example.com" for _ in range(N_RECORDS)]
    data['CIN'] = [f"{random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}{random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}{random.randint(100000,999999)}" for _ in range(N_RECORDS)]

    ordered_columns = [
        'Age', 'Categorie_age', 'Sexe', 'Milieu', 'Region_geographique', 'Etat_matrimonial',
        'Niveau_education', 'Annees_experience', 'CSP', 'Secteur_emploi',
        'Propriete_immobiliere', 'Vehicule_motorise', 'Terrain_agricole',
        'Revenu_secondaire', 'Revenu_Annuel', 'Revenu_Mensuel',  # Ajout de Revenu_Mensuel
        'Adresse_Email', 'CIN'  # Ajout de Adresse_Email et CIN
    ]
    final_columns = [col for col in ordered_columns if col in data.columns]
    missing_cols = [col for col in data.columns if col not in final_columns] # Au cas o?
    data = data[final_columns + missing_cols]

    return data

# --- Main Execution ---
if __name__ == "__main__":
    print(f"G?n?ration de {N_RECORDS} enregistrements avec le script ...")
    dataset = generate_dataset()
    
    print("\n--- Statistiques Descriptives du Revenu Annuel  ---")
    print(dataset['Revenu_Annuel'].describe())
    
    # --- V?rification des Contraintes HCP ---
    print("\n--- V?rification des Contraintes Statistiques (Bas?es sur HCP) ---")
    
    # Revenus moyens globaux et par milieu
    revenu_moyen_global_genere = dataset['Revenu_Annuel'].mean()
    print(f"Revenu Annuel Moyen Global (G?n?r?): {revenu_moyen_global_genere:.0f} DH (Cible HCP: 21949 DH)")

    moyennes_milieu_genere = dataset.groupby('Milieu')['Revenu_Annuel'].mean()
    revenu_moyen_urbain_genere = moyennes_milieu_genere.get('Urbain', np.nan)
    revenu_moyen_rural_genere = moyennes_milieu_genere.get('Rural', np.nan)

    print(f"Revenu Annuel Moyen Urbain (G?n?r?): {revenu_moyen_urbain_genere:.0f} DH (Cible HCP: 26988 DH)")
    print(f"Revenu Annuel Moyen Rural (G?n?r?): {revenu_moyen_rural_genere:.0f} DH (Cible HCP: 12862 DH)")

    # Proportions de revenus inf?rieurs ? la moyenne
    if pd.notna(revenu_moyen_global_genere):
        prop_inf_moy_global = (dataset['Revenu_Annuel'] < revenu_moyen_global_genere).mean() * 100
        print(f"Proportion revenus < moyenne globale (G?n?r?): {prop_inf_moy_global:.1f}% (Cible HCP: 71.8%)")
    else:
        print("Proportion revenus < moyenne globale (G?n?r?): Non calculable (moyenne globale NaN)")

    if pd.notna(revenu_moyen_urbain_genere) and 'Urbain' in moyennes_milieu_genere:
        prop_inf_moy_urbain = (dataset[dataset['Milieu'] == 'Urbain']['Revenu_Annuel'] < revenu_moyen_urbain_genere).mean() * 100
        print(f"Proportion revenus < moyenne urbaine (G?n?r?): {prop_inf_moy_urbain:.1f}% (Cible HCP: 65.9%)")
    else:
        print("Proportion revenus < moyenne urbaine (G?n?r?): Non calculable")
    
    if pd.notna(revenu_moyen_rural_genere) and 'Rural' in moyennes_milieu_genere:
        prop_inf_moy_rural = (dataset[dataset['Milieu'] == 'Rural']['Revenu_Annuel'] < revenu_moyen_rural_genere).mean() * 100
        print(f"Proportion revenus < moyenne rurale (G?n?r?): {prop_inf_moy_rural:.1f}% (Cible HCP: 85.4%)")
    else:
        print("Proportion revenus < moyenne rurale (G?n?r?): Non calculable")

    print("\nMoyennes par CSP (G?n?r?) :")
    print(dataset.groupby('CSP')['Revenu_Annuel'].mean().sort_values(ascending=False))

    print("\nMoyennes par Niveau d'?ducation (G?n?r?) :")
    print(dataset.groupby('Niveau_education')['Revenu_Annuel'].mean().sort_values(ascending=False))

    dataset.to_csv(FILENAME, index=False, encoding='utf-8')
    print(f"\nDataset '{FILENAME}' g?n?r? avec succ?s ({len(dataset)} enregistrements).")


In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
FILENAME = "dataset_revenu_marocains.csv"
RANDOM_SEED = 42  # Also needed for train_test_split

def load_and_preprocess_data(file_path="dataset_revenu_marocains.csv"):
    """
    Charge et pr?traite le dataset pour la mod?lisation.
    Retourne les ensembles de donn?es et un dictionnaire des objets de pr?traitement.
    """
    # 1. Charger le dataset
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Erreur : Le fichier '{file_path}' n'a pas ?t? trouv?.")
        return None, None, None, None, None 
    
    print("Dataset charg? avec succ?s pour le pr?traitement.")

    # 2. S?parer X et y
    if 'Revenu_Annuel' not in df.columns:
        print("Erreur : La colonne 'Revenu_Annuel' (cible) est manquante dans le dataset.")
        return None, None, None, None, None

    y = df['Revenu_Annuel'].copy()
    cols_to_drop_for_X = ['Revenu_Annuel', 'Revenu_Mensuel', 'Adresse_Email', 'CIN', 'Categorie_age']
    X = df.drop(columns=[col for col in cols_to_drop_for_X if col in df.columns], errors='ignore')
    print("Variable cible et caract?ristiques s?par?es.")

    # 3. Diviser en ensembles d'entra?nement et de test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
    print(f"Donn?es divis?es : X_train shape {X_train.shape}, X_test shape {X_test.shape}")

    X_train = X_train.copy()
    X_test = X_test.copy()

    # 4. Identifier les types de colonnes originales ? partir de X_train
    # Ces listes sont captur?es AVANT toute imputation ou transformation des colonnes.
    original_numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    original_categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"Colonnes num?riques originales identifi?es : {original_numerical_cols}")
    print(f"Colonnes cat?gorielles originales identifi?es : {original_categorical_cols}")

    # 5. Imputation des valeurs manquantes
    num_imputer = SimpleImputer(strategy='median')
    cat_imputer = SimpleImputer(strategy='most_frequent')

    if original_numerical_cols:
        X_train[original_numerical_cols] = num_imputer.fit_transform(X_train[original_numerical_cols])
        X_test[original_numerical_cols] = num_imputer.transform(X_test[original_numerical_cols])
        print("Imputation des valeurs manquantes num?riques termin?e (m?diane).")

    if original_categorical_cols:
        X_train[original_categorical_cols] = cat_imputer.fit_transform(X_train[original_categorical_cols])
        X_test[original_categorical_cols] = cat_imputer.transform(X_test[original_categorical_cols])
        print("Imputation des valeurs manquantes cat?gorielles termin?e (mode).")

    # 6. Encodage One-Hot des variables cat?gorielles originales
    train_cols_final_list = [] # Initialiser la liste des colonnes finales
    if original_categorical_cols:
        X_train = pd.get_dummies(X_train, columns=original_categorical_cols, prefix=original_categorical_cols, dummy_na=False)
        X_test = pd.get_dummies(X_test, columns=original_categorical_cols, prefix=original_categorical_cols, dummy_na=False)
        print("Encodage One-Hot termin?.")

        train_cols_final_list = X_train.columns.tolist() # Colonnes apr?s one-hot
        X_test = X_test.reindex(columns=train_cols_final_list, fill_value=0)
        X_train = X_train[train_cols_final_list] # S'assurer de l'ordre et de l'ensemble des colonnes
        print("Colonnes align?es entre les ensembles d'entra?nement et de test.")
        print(f"Nouvelles dimensions apr?s encodage : X_train shape {X_train.shape}, X_test shape {X_test.shape}")
    else:
        # S'il n'y a pas de colonnes cat?gorielles, les colonnes finales sont celles de X_train actuelles
        train_cols_final_list = X_train.columns.tolist()

    # 7. Standardisation des variables num?riques originales
    # Les noms des colonnes num?riques originales sont pr?serv?s apr?s le one-hot encoding des cat?gorielles.
    scaler = StandardScaler()
    if original_numerical_cols:
        # S'assurer que les colonnes ? standardiser sont bien pr?sentes dans X_train (elles devraient l'?tre)
        cols_to_scale = [col for col in original_numerical_cols if col in X_train.columns]
        if cols_to_scale:
            X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
            X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])
            print("Standardisation des colonnes num?riques d'origine termin?e.")
        else:
            print("Aucune colonne num?rique (parmi les originales) ? standardiser trouv?e dans X_train apr?s encodage.")
            
    # Imputation de y_train et y_test si elles contiennent des NaN
    y_imputer = SimpleImputer(strategy='median')
    if y_train.isnull().any():
        y_train = pd.Series(y_imputer.fit_transform(y_train.values.reshape(-1, 1)).flatten(), name=y_train.name, index=y_train.index)
        print("Imputation des NaN dans y_train termin?e.")
    if y_test.isnull().any():
        y_test = pd.Series(y_imputer.transform(y_test.values.reshape(-1, 1)).flatten(), name=y_test.name, index=y_test.index)
        print("Imputation des NaN dans y_test termin?e.")

    print("Pr?traitement termin?.")
    
    preprocessor_objects = {
        'num_imputer': num_imputer,
        'cat_imputer': cat_imputer,
        'scaler': scaler,
        'train_cols_after_dummies': train_cols_final_list,       # Colonnes finales vues par le mod?le
        'original_numerical_cols': original_numerical_cols,     # Colonnes num?riques avant toute transfo
        'original_categorical_cols': original_categorical_cols  # Colonnes cat?gorielles avant toute transfo
    }
    return X_train, X_test, y_train, y_test, preprocessor_objects

if __name__ == "__main__":
    # --- Pr?traitement des donn?es ---
    print("\n--- Lancement du Pr?traitement des Donn?es ---")
    X_train, X_test, y_train, y_test, preprocessor_objects = load_and_preprocess_data(FILENAME)

    if X_train is not None and preprocessor_objects is not None:
        print("\n--- Informations sur les donn?es pr?trait?es ---")
        print("X_train shape:", X_train.shape)
        print("X_test shape:", X_test.shape)
        print("y_train shape:", y_train.shape)
        print("y_test shape:", y_test.shape)
        
        print("\nPremi?res lignes de X_train (apr?s pr?traitement):")
        print(X_train.head())
        
        print("\nStatistiques de y_train:")
        print(y_train.describe())

        print("\n--- Composants du pr?processeur sauvegard?s ---")
        for key, val in preprocessor_objects.items():
            if hasattr(val, 'shape'):
                 print(f"{key}: type {type(val)}, shape {val.shape if isinstance(val, np.ndarray) else 'N/A'}")
            elif isinstance(val, list):
                 print(f"{key}: type {type(val)}, length {len(val)}, exemple {val[:3] if len(val) > 0 else '[]'}")
            else:
                 print(f"{key}: type {type(val)}")
    else:
        print("?chec du pr?traitement des donn?es.")


In [ ]:
# CELLULE 3 : Importations suppl?mentaires et initialisation pour la mod?lisation

import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

# Pour stocker les performances des mod?les
model_performance = {}

# D?finir une constante pour la reproductibilit? si n?cessaire dans les mod?les
RANDOM_STATE = 42

# V?rifier si les donn?es sont charg?es (elles devraient l'?tre par la cellule pr?c?dente)
if 'X_train' not in locals() or \
   'X_test' not in locals() or \
   'y_train' not in locals() or \
   'y_test' not in locals():
    print("ERREUR : Les donn?es X_train, X_test, y_train, y_test ne sont pas charg?es.")
    print("Veuillez ex?cuter la cellule de pr?traitement des donn?es (load_and_preprocess_data).")
else:
    print("Donn?es d'entra?nement et de test pr?tes pour la mod?lisation.")
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

In [ ]:
# CELLULE 4 : 3.1. R?gression Lin?aire (Mod?le de Base)

if 'X_train' in locals():
    print("\n--- 3.1. Entra?nement du Mod?le de R?gression Lin?aire ---")
    
    # Initialiser et entra?ner le mod?le
    linear_model = LinearRegression()
    
    start_time_lr = time.time()
    linear_model.fit(X_train, y_train)
    end_time_lr = time.time()
    
    training_time_lr = end_time_lr - start_time_lr
    print(f"Entra?nement termin? en {training_time_lr:.2f} secondes.")
    
    # Pr?dictions
    predictions_lr = linear_model.predict(X_test)
    
    # ?valuation
    mae_lr = mean_absolute_error(y_test, predictions_lr)
    rmse_lr = np.sqrt(mean_squared_error(y_test, predictions_lr))
    r2_lr = r2_score(y_test, predictions_lr)
    
    print("\n--- ?valuation du Mod?le de R?gression Lin?aire ---")
    print(f"MAE (Mean Absolute Error): {mae_lr:.2f} DH")
    print(f"RMSE (Root Mean Squared Error): {rmse_lr:.2f} DH")
    print(f"R? (Coefficient de d?termination): {r2_lr:.4f}")
    
    # Stocker les performances
    model_performance['Linear Regression'] = {
        'MAE': mae_lr,
        'RMSE': rmse_lr,
        'R2': r2_lr,
        'Training Time (s)': training_time_lr,
        'Best Params': 'N/A'
    }
    
    print("\n--- Aper?u des Pr?dictions vs Valeurs R?elles (R?gression Lin?aire) ---")
    df_predictions_lr = pd.DataFrame({'R?el': y_test.values, 'Pr?dit_LR': predictions_lr})
    display(df_predictions_lr.head(20))
    # Visualisation des r?sultats
    import matplotlib.pyplot as plt


    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, predictions_lr, alpha=0.7)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.title('Pr?dictions vs Valeurs R?elles')
    plt.xlabel('Valeurs R?elles')
    plt.ylabel('Pr?dictions')
    plt.grid()
    plt.show()
    
    # Calcul des r?sidus
    residuals = y_test - predictions_lr

    # Visualisation des r?sidus
    plt.figure(figsize=(10, 5))
    sns.histplot(residuals, kde=True, bins=50)
    plt.title("Distribution des r?sidus")
    plt.show()
else:
    print("Veuillez d'abord charger et pr?traiter les donn?es.")


In [ ]:
# CELLULE 5 : 3.2. Arbres de D?cision (DecisionTreeRegressor) avec Ajustement des Hyperparam?tres

if 'X_train' in locals():
    print("\n--- 3.2. Entra?nement et Ajustement des Hyperparam?tres pour DecisionTreeRegressor ---")

    # D?finir la grille d'hyperparam?tres selon les sp?cifications du projet
    param_grid_dt = {
        'criterion': ['squared_error'], # 'friedman_mse' est aussi une option
        'max_depth': [5],
        'min_samples_split': [2]
    }
    # Note: 'absolute_error' pour criterion peut ?tre significativement plus lent.

    # Initialiser GridSearchCV
    grid_search_dt = GridSearchCV(
        estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
        param_grid=param_grid_dt,
        cv=5, # Nombre de plis pour la validation crois?e
        scoring='neg_mean_squared_error', # M?trique pour l'?valuation
        n_jobs=-1, # Utiliser tous les processeurs disponibles
        verbose=1 # Afficher des messages pendant l'ajustement
    )

    print(f"Ajustement des hyperparam?tres pour DecisionTreeRegressor en cours (CV={grid_search_dt.cv})...")
    start_time_dt_grid = time.time()
    grid_search_dt.fit(X_train, y_train)
    end_time_dt_grid = time.time()
    grid_search_time_dt = end_time_dt_grid - start_time_dt_grid
    print(f"Ajustement termin? en {grid_search_time_dt:.2f} secondes.")

    # Meilleurs hyperparam?tres et score
    print(f"\nMeilleurs hyperparam?tres trouv?s pour DecisionTreeRegressor : {grid_search_dt.best_params_}")
    # Le best_score_ est neg_mean_squared_error, donc plus proche de 0 est mieux.
    print(f"Meilleur score (neg_mean_squared_error) de validation crois?e : {grid_search_dt.best_score_:.4f}")
    
    # Meilleur estimateur
    best_dt_model = grid_search_dt.best_estimator_
    
    # Pr?dictions sur l'ensemble de test avec le meilleur mod?le
    predictions_dt = best_dt_model.predict(X_test)
    
    # ?valuation du meilleur mod?le
    mae_dt = mean_absolute_error(y_test, predictions_dt)
    rmse_dt = np.sqrt(mean_squared_error(y_test, predictions_dt))
    r2_dt = r2_score(y_test, predictions_dt)
    
    print("\n--- ?valuation du Mod?le DecisionTreeRegressor (avec meilleurs hyperparam?tres) ---")
    print(f"MAE: {mae_dt:.2f} DH")
    print(f"RMSE: {rmse_dt:.2f} DH")
    print(f"R?: {r2_dt:.4f}")
    
    # Stocker les performances
    model_performance['Decision Tree (Tuned)'] = {
        'MAE': mae_dt,
        'RMSE': rmse_dt,
        'R2': r2_dt,
        'Training Time (s)': grid_search_time_dt, # Temps pour GridSearchCV
        'Best Params': grid_search_dt.best_params_
    }

    print("\n--- Aper?u des Pr?dictions vs Valeurs R?elles (Decision Tree) ---")
    df_predictions_dt = pd.DataFrame({'R?el': y_test.values, 'Pr?dit_DT': predictions_dt})
    display(df_predictions_dt.head(20))
else:
    print("Veuillez d'abord charger et pr?traiter les donn?es.")


In [ ]:
# CELLULE 7 : 3.3. For?ts Al?atoires (RandomForestRegressor) avec Ajustement des Hyperparam?tres

if 'X_train' in locals():
    print("\n--- 3.3. Entra?nement et Ajustement des Hyperparam?tres pour RandomForestRegressor ---")
    from sklearn.ensemble import RandomForestRegressor

    # D?finir une grille d'hyperparam?tres plus cibl?e pour RandomForest
    # RandomForest peut ?tre long ? entra?ner avec une grille large.
    param_grid_rf = {
        'n_estimators': [50, 100, 150, 200], # Nombre d'arbres dans la for?t
        'criterion': ['squared_error'], # 'absolute_error' est tr?s lent pour RF, 'friedman_mse' est une alternative
        'max_depth': [None, 5, 10, 15, 20],      # Profondeur max des arbres (similaire ? DT)
        # 'min_samples_split': [2, 4], # Nombre min d'?chantillons pour diviser un n?ud
        # 'min_samples_leaf': [1, 2], # Nombre min d'?chantillons par feuille (peut ?tre ajout?)
        # 'max_features': ['sqrt', 'log2'] # Nombre de features ? consid?rer pour la meilleure division (peut ?tre ajout?)
    }
    # Pour un premier essai, cette grille est raisonnable.
    # Si le temps le permet, vous pouvez l'?tendre.

    # Initialiser GridSearchCV
    grid_search_rf = GridSearchCV(
        estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), # n_jobs dans RF pour l'entra?nement des arbres
        param_grid=param_grid_rf,
        cv=5, # R?duire le nombre de plis pour acc?l?rer si n?cessaire (5 est standard)
        scoring='neg_mean_squared_error',
        n_jobs=-1, # n_jobs dans GridSearchCV pour les plis de CV
        verbose=1
    )

    print(f"Ajustement des hyperparam?tres pour RandomForestRegressor en cours (CV={grid_search_rf.cv})...")
    start_time_rf_grid = time.time()
    grid_search_rf.fit(X_train, y_train)
    end_time_rf_grid = time.time()
    grid_search_time_rf = end_time_rf_grid - start_time_rf_grid
    print(f"Ajustement termin? en {grid_search_time_rf:.2f} secondes.")

    # Meilleurs hyperparam?tres et score
    print(f"\nMeilleurs hyperparam?tres trouv?s pour RandomForestRegressor : {grid_search_rf.best_params_}")
    print(f"Meilleur score (neg_mean_squared_error) de validation crois?e : {grid_search_rf.best_score_:.4f}")
    
    # Meilleur estimateur
    best_rf_model = grid_search_rf.best_estimator_
    
    # Pr?dictions
    predictions_rf = best_rf_model.predict(X_test)
    
    # ?valuation
    mae_rf = mean_absolute_error(y_test, predictions_rf)
    rmse_rf = np.sqrt(mean_squared_error(y_test, predictions_rf))
    r2_rf = r2_score(y_test, predictions_rf)
    
    print("\n--- ?valuation du Mod?le RandomForestRegressor (avec meilleurs hyperparam?tres) ---")
    print(f"MAE: {mae_rf:.2f} DH")
    print(f"RMSE: {rmse_rf:.2f} DH")
    print(f"R?: {r2_rf:.4f}")
    
    # Stocker les performances
    model_performance['Random Forest (Tuned)'] = {
        'MAE': mae_rf,
        'RMSE': rmse_rf,
        'R2': r2_rf,
        'Training Time (s)': grid_search_time_rf,
        'Best Params': grid_search_rf.best_params_
    }

    print("\n--- Aper?u des Pr?dictions vs Valeurs R?elles (Random Forest) ---")
    df_predictions_rf = pd.DataFrame({'R?el': y_test.values, 'Pr?dit_RF': predictions_rf})
    display(df_predictions_rf.head(20))

    # Mettre ? jour et afficher le tableau comparatif
    print("\n--- Tableau Comparatif des Performances des Mod?les (mis ? jour) ---")
    performance_df = pd.DataFrame.from_dict(model_performance, orient='index')
    display(performance_df.sort_values(by='RMSE', ascending=True))

else:
    print("Veuillez d'abord charger et pr?traiter les donn?es.")


In [ ]:
# CELLULE 8 : 3.4. Gradient Boosting Regressor avec Ajustement des Hyperparam?tres

if 'X_train' in locals():
    print("\n--- 3.4. Entra?nement et Ajustement des Hyperparam?tres pour GradientBoostingRegressor ---")
    from sklearn.ensemble import GradientBoostingRegressor

    # D?finir une grille d'hyperparam?tres pour GradientBoostingRegressor
    # Gradient Boosting peut aussi ?tre long ? entra?ner.
    param_grid_gbr = {
        'loss': ['squared_error'], # Ajout de 'absolute_error'
        'n_estimators': [100, 200, 300],          
        'learning_rate': [0.01, 0.1, 0.2],                
        'subsample': [0.5, 0.8, 1.0]            
    }
    # Cette grille est un bon point de d?part.

    # Initialiser GridSearchCV
    grid_search_gbr = GridSearchCV(
        estimator=GradientBoostingRegressor(random_state=RANDOM_STATE),
        param_grid=param_grid_gbr,
        cv=3, # Maintenir cv=3 pour la vitesse, augmenter ? 5 si le temps le permet
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    print(f"Ajustement des hyperparam?tres pour GradientBoostingRegressor en cours (CV={grid_search_gbr.cv})...")
    start_time_gbr_grid = time.time()
    grid_search_gbr.fit(X_train, y_train)
    end_time_gbr_grid = time.time()
    grid_search_time_gbr = end_time_gbr_grid - start_time_gbr_grid
    print(f"Ajustement termin? en {grid_search_time_gbr:.2f} secondes.")

    # Meilleurs hyperparam?tres et score
    print(f"\nMeilleurs hyperparam?tres trouv?s pour GradientBoostingRegressor : {grid_search_gbr.best_params_}")
    print(f"Meilleur score (neg_mean_squared_error) de validation crois?e : {grid_search_gbr.best_score_:.4f}")
    
    # Meilleur estimateur
    best_gbr_model = grid_search_gbr.best_estimator_
    
    # Pr?dictions
    predictions_gbr = best_gbr_model.predict(X_test)
    
    # ?valuation
    mae_gbr = mean_absolute_error(y_test, predictions_gbr)
    rmse_gbr = np.sqrt(mean_squared_error(y_test, predictions_gbr))
    r2_gbr = r2_score(y_test, predictions_gbr)
    
    print("\n--- ?valuation du Mod?le GradientBoostingRegressor (avec meilleurs hyperparam?tres) ---")
    print(f"MAE: {mae_gbr:.2f} DH")
    print(f"RMSE: {rmse_gbr:.2f} DH")
    print(f"R?: {r2_gbr:.4f}")
    
    # Stocker les performances
    model_performance['Gradient Boosting (Tuned)'] = {
        'MAE': mae_gbr,
        'RMSE': rmse_gbr,
        'R2': r2_gbr,
        'Training Time (s)': grid_search_time_gbr,
        'Best Params': grid_search_gbr.best_params_
    }

    print("\n--- Aper?u des Pr?dictions vs Valeurs R?elles (Gradient Boosting) ---")
    df_predictions_gbr = pd.DataFrame({'R?el': y_test.values, 'Pr?dit_GBR': predictions_gbr})
    display(df_predictions_gbr.head(20))

    # Mettre ? jour et afficher le tableau comparatif
    print("\n--- Tableau Comparatif des Performances des Mod?les (mis ? jour) ---")
    performance_df = pd.DataFrame.from_dict(model_performance, orient='index')
    display(performance_df.sort_values(by='RMSE', ascending=True))

else:
    print("Veuillez d'abord charger et pr?traiter les donn?es.")


In [ ]:
# CELLULE 9 : 3.5. R?seaux de Neurones (MLPRegressor) avec Ajustement des Hyperparam?tres

if 'X_train' in locals():
    print("\n--- 3.5. Entra?nement et Ajustement des Hyperparam?tres pour MLPRegressor ---")
    from sklearn.neural_network import MLPRegressor
    from sklearn.preprocessing import StandardScaler # S'assurer que les donn?es sont bien standardis?es pour MLP

    # Note: Les donn?es X_train, X_test devraient d?j? ?tre standardis?es par preprocess_data_.py
    # Si ce n'?tait pas le cas, il faudrait le faire ici sp?cifiquement pour MLP.

    # D?finir une grille d'hyperparam?tres TR?S SIMPLE pour MLPRegressor pour un premier essai
    # L'exploration d'architectures de r?seau peut ?tre tr?s longue.
    param_grid_mlp = { # Remplacer par la grille simplifi?e
        'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100)],
        'activation': ['relu'],
        'solver': ['adam'],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate_init': [0.001, 0.01],
        'max_iter': [300, 500], # S'assurer que early_stopping est activ? dans l'estimateur
        # 'batch_size': [32, 64] # Pourrait ?tre test? dans une seconde passe
    }

    grid_search_mlp = GridSearchCV(
        estimator=MLPRegressor(random_state=RANDOM_STATE, early_stopping=True, n_iter_no_change=15, tol=1e-4), # Ajuster n_iter_no_change et tol
        param_grid=param_grid_mlp,
        cv=2, # Maintenir cv=2 ou 3 pour la vitesse
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    print(f"Ajustement des hyperparam?tres pour MLPRegressor en cours (CV={grid_search_mlp.cv})...")
    print("Cela peut prendre beaucoup de temps...")
    start_time_mlp_grid = time.time()
    grid_search_mlp.fit(X_train, y_train) # Utiliser X_train_scaled
    end_time_mlp_grid = time.time()
    grid_search_time_mlp = end_time_mlp_grid - start_time_mlp_grid
    print(f"Ajustement termin? en {grid_search_time_mlp:.2f} secondes.")

    # Meilleurs hyperparam?tres et score
    print(f"\nMeilleurs hyperparam?tres trouv?s pour MLPRegressor : {grid_search_mlp.best_params_}")
    print(f"Meilleur score (neg_mean_squared_error) de validation crois?e : {grid_search_mlp.best_score_:.4f}")
    
    # Meilleur estimateur
    best_mlp_model = grid_search_mlp.best_estimator_
    
    # Pr?dictions
    predictions_mlp = best_mlp_model.predict(X_test) # Utiliser X_test_scaled
    
    # ?valuation
    mae_mlp = mean_absolute_error(y_test, predictions_mlp)
    rmse_mlp = np.sqrt(mean_squared_error(y_test, predictions_mlp))
    r2_mlp = r2_score(y_test, predictions_mlp)
    
    print("\n--- ?valuation du Mod?le MLPRegressor (avec meilleurs hyperparam?tres) ---")
    print(f"MAE: {mae_mlp:.2f} DH")
    print(f"RMSE: {rmse_mlp:.2f} DH")
    print(f"R?: {r2_mlp:.4f}")
    
    # Stocker les performances
    model_performance['MLP Regressor (Tuned)'] = {
        'MAE': mae_mlp,
        'RMSE': rmse_mlp,
        'R2': r2_mlp,
        'Training Time (s)': grid_search_time_mlp,
        'Best Params': grid_search_mlp.best_params_
    }

    print("\n--- Aper?u des Pr?dictions vs Valeurs R?elles (MLP Regressor) ---")
    df_predictions_mlp = pd.DataFrame({'R?el': y_test.values, 'Pr?dit_MLP': predictions_mlp})
    display(df_predictions_mlp.head(20))

    # Mettre ? jour et afficher le tableau comparatif
    print("\n--- Tableau Comparatif des Performances des Mod?les (mis ? jour) ---")
    performance_df = pd.DataFrame.from_dict(model_performance, orient='index')
    display(performance_df.sort_values(by='RMSE', ascending=True))

else:
    print("Veuillez d'abord charger et pr?traiter les donn?es.")


In [ ]:
# CELLULE 6 : Affichage du Tableau Comparatif des Performances

if model_performance:
    print("\n--- Tableau Comparatif des Performances des Mod?les ---")
    performance_df = pd.DataFrame.from_dict(model_performance, orient='index')
    display(performance_df.sort_values(by='RMSE', ascending=True))
else:
    print("Aucun mod?le n'a encore ?t? entra?n? et ?valu?.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisation de la distribution de y_train AVANT transformation
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(y_train, kde=True)
plt.title("Distribution de y_train (Revenu Annuel) AVANT transformation")
plt.xlabel("Revenu Annuel")
plt.ylabel("Fr?quence")

# Application de la transformation log1p
y_train_transformed = np.log1p(y_train)
y_test_transformed = np.log1p(y_test)

# Visualisation de la distribution de y_train APRES transformation
plt.subplot(1, 2, 2)
sns.histplot(y_train_transformed, kde=True, color='green')
plt.title("Distribution de y_train_transformed (log(1+Revenu)) APRES transformation")
plt.xlabel("log(1 + Revenu Annuel)")
plt.ylabel("Fr?quence")
plt.tight_layout()
plt.show()

print("\n--- Aper?u de y_train_transformed (quelques valeurs) ---")
print(y_train_transformed.head())

In [ ]:

import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import time

# Constants
RANDOM_STATE = 42

# Ensure the CustomPreprocessor class definition is REMOVED from this cell if it was present.

def create_best_mlp(X_train, y_train, X_test, y_test, preprocessor_objects_dict):
    """
    Create and evaluate MLP model with known best parameters.
    preprocessor_objects_dict should contain all necessary fitted preprocessing components
    and lists of original numerical/categorical column names, and final training column names.
    """
    
    print("\n=== Creating MLP Regressor with Best Known Parameters ===")
    
    # Use your previously determined best parameters
    best_params = {
        'hidden_layer_sizes': (100, 100),
        'activation': 'relu',
        'solver': 'adam',
        'alpha': 0.001,
        'learning_rate_init': 0.01,
        'max_iter': 300,
        'early_stopping': True,
        'random_state': RANDOM_STATE,
        'n_iter_no_change': 15,
        'tol': 1e-4,
        'batch_size': 128
    }
    
    # Initialize and train MLP with best parameters
    print("Training MLP with best parameters...")
    start_time = time.time()
    best_mlp = MLPRegressor(**best_params)
    best_mlp.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Evaluate on test set
    y_pred = best_mlp.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print("\n=== Model Performance ===")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Test MAE: {mae:.2f} DH")
    print(f"Test RMSE: {rmse:.2f} DH")
    print(f"Test R?: {r2:.4f}")
    
    # Save the model along with the dictionary of preprocessor components
    # X_train.columns.tolist() provides the list of feature names as seen by the model during training
    save_model(best_mlp, preprocessor_objects_dict, X_train.columns.tolist(), 
               {'MAE': mae, 'RMSE': rmse, 'R2': r2}, best_params)
    
    return best_mlp


def save_model(model, preprocessor_components_dict, trained_feature_names, metrics, best_params):
    """
    Save the trained model with all necessary preprocessing components (as a dictionary)
    and metadata.
    """
    model_data = {
        'model': model,
        'preprocessor_components': preprocessor_components_dict, # Store the dictionary of components
        'metadata': {
            'performance': metrics,
            'trained_feature_names': trained_feature_names, # List of column names after all preprocessing
            'best_params': best_params,
            'timestamp': pd.Timestamp.now().isoformat()
        }
    }
    
    joblib.dump(model_data, 'best_mlp_model.pkl')
    print("\nModel saved as 'best_mlp_model.pkl'")


# Main execution
# Ensure 'preprocessor_objects' (the dictionary from load_and_preprocess_data) is available
# and correctly populated with all required keys.
if 'X_train' in locals() and 'y_train' in locals() and \
   'X_test' in locals() and 'y_test' in locals() and \
   'preprocessor_objects' in locals(): # preprocessor_objects is the dictionary
    
    # Ensure preprocessor_objects contains all required keys before calling create_best_mlp
    required_keys = ['num_imputer', 'cat_imputer', 'scaler', 'train_cols_after_dummies', 'original_numerical_cols', 'original_categorical_cols']
    if not all(key in preprocessor_objects for key in required_keys):
        print(f"Error: 'preprocessor_objects' dictionary is missing one or more required keys: {required_keys}")
        # Handle error appropriately, e.g., skip model creation or raise an exception
    else:
        best_mlp_model = create_best_mlp(X_train, y_train, X_test, y_test, preprocessor_objects)
        
        sample_preds = pd.DataFrame({
            'Actual': y_test[:20],
            'Predicted': best_mlp_model.predict(X_test[:20]) # X_test is already preprocessed here
        })
        print("\nSample predictions:")
        print(sample_preds)
else:
    print("Error: Required data (X_train, y_train, X_test, y_test, preprocessor_objects) not found. Please run preprocessing first.")
